In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from plotting import *
setup_plotting_style()
print(f"saving figures:{SAVE_FIGURES}")

In [ ]:
import numpy as np
import pandas as pd
from scipy import signal
import scipy.stats as stats
from scipy.stats import gmean
from matplotlib.pylab import norm


NB_ID = "21"

subset_samples = 1000

In [ ]:
def load_barrage_data():
    
    print(f"Loading {MIT_BARRAGE_X.name}...")
    X = np.load(MIT_BARRAGE_X, mmap_mode='r')
    Y = np.load(MIT_BARRAGE_Y, mmap_mode='r')
    df = pd.read_csv(MIT_BARRAGE_DATASET_METADATA_FILE)
    
    return X, Y, df

X_barr, Y_barr, df_meta = load_barrage_data()

print(f"--- Loaded ---")
print(f"Mixtures: {X_barr.shape}")
print(f"Targets:  {Y_barr.shape}")
print(f"Metadata: {len(df_meta)} rows")

In [ ]:
df_subset = df_meta.sample(n=min(subset_samples, len(df_meta)), random_state=42)

plt.figure(figsize=(10, 5))

ax = sns.histplot(data=df_subset, x='sinr_db', bins=15, kde=True, color='C0')

ax.set_title(f"SINR Distribution (Subset n={len(df_subset)})")
ax.set_xlabel("SINR (dB)")
ax.set_ylabel("Sample Density")

# Save and show
save_plot("sinr_distribution_subset", machine_id="B", nb_id=NB_ID, fig_id="01")
plt.show()

# --- VERIFICATION (On Full Metadata) ---
# Reading the CSV-based metadata is fast
print(f"Total Samples in Metadata: {len(df_meta)}")
print(f"SINR Range: {df_meta['sinr_db'].min():.2f} dB to {df_meta['sinr_db'].max():.2f} dB")

# Balance Check: For continuous data, we check the standard deviation of bins
counts, _ = np.histogram(df_meta['sinr_db'], bins=10)
print(f"Balance Check: Samples per bin (approx): {counts}")
print(f"Max deviation from average: {np.abs(counts - np.mean(counts)).max():.1f} samples")

# Fig 21.1 Statistical Balance: Dataset Composition

The Statistical Balance analysis performs the first critical sanity check: ensuring the dataset provides equal learning opportunities across all difficulty levels.

### The "Flat" Distribution
* **Result:** The distribution is uniform, with approximately equal sample counts across all six SINR levels (from -12 dB to +3 dB).
* **Implication:** The dataset is not biased towards "easy" or "hard" examples. This ensures that during training, the Autoencoder will not overfit to a specific noise level and will be forced to learn robust features across the entire difficulty spectrum. No class weighting will be required for the loss function.

In [ ]:
subset_n = min(subset_samples, len(df_meta))
indices = np.random.choice(len(df_meta), size=subset_n, replace=False)

# Fetch ONLY the subset into RAM from the disk-mapped arrays
X_sub = X_barr[indices]
Y_sub = Y_barr[indices]
meta_sub = df_meta.iloc[indices]

# Perform math only on the 1000 samples
Noise_est_sub = X_sub - Y_sub

# Calculate Power
P_clean_sub = np.mean(np.abs(Y_sub)**2, axis=1)
P_noise_sub = np.mean(np.abs(Noise_est_sub)**2, axis=1)

# Measured SINR
sinr_measured_sub = 10 * np.log10(P_clean_sub / (P_noise_sub + 1e-12))

# Plot the Subset
plt.figure(figsize=(6, 6))
sns.scatterplot(x=meta_sub['sinr_db'], y=sinr_measured_sub, alpha=0.5)

# Calculate axis limits for the 'Ideal' line
min_val = min(meta_sub['sinr_db'].min(), sinr_measured_sub.min())
max_val = max(meta_sub['sinr_db'].max(), sinr_measured_sub.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Ideal')

plt.title(f"Target vs Measured SINR (Barrage Subset: n={subset_n})")
plt.xlabel("Target SINR (dB)")
plt.ylabel("Measured SINR (dB)")
plt.legend()
plt.grid(True)

save_plot("barrage_sinr_physics_check", machine_id="B", nb_id=NB_ID, fig_id="02")
plt.show()

# Error calculation on the subset
error_sub = np.mean(np.abs(meta_sub['sinr_db'] - sinr_measured_sub))
print(f"Mean SINR Error (on {subset_n} samples): {error_sub:.4f} dB")
assert error_sub < 0.1, f"CRITICAL: Barrage mixing math is wrong! Error: {error_sub:.4f} dB"

# Fig 21.2 Physics Check: The "Mixing Engine"

This scatter plot validates the correction of the "Unit Power Assumption" bug found in previous sessions.

### The Data
* **X-Axis:** `sinr_target` (The requested noise level).
* **Y-Axis:** `sinr_measured` (The actual calculated noise level).
* **Red Line:** The ideal $y=x$ identity line.

### The Verdict: "Perfect Alignment" 
* **Observation:** The blue dots align tightly with the red reference line.
* **Physics Explanation:**
    * **The Fix:** By dynamically calculating the power of every slice ($P_{signal}$) before mixing, we eliminated the ~2dB drift observed in the MIT dataset.
    * **Precision:** The system now delivers the exact "Harshness" requested. A label of -10dB is physically accurate, ensuring the Autoencoder learns valid denoising gradients.

In [ ]:
def calc_sfm(signals):
    sfm_list = []
    for sig in signals:
        # Welch's PSD
        f, psd = signal.welch(sig, fs=1.0, nperseg=1024)
        psd = psd[1:] # Remove DC
        
        # SFM = Geometric Mean / Arithmetic Mean
        sfm = gmean(psd) / (np.mean(psd) + 1e-12)
        sfm_list.append(sfm)
    return np.array(sfm_list)

print(f"Calculating Spectral Flatness for a subset (first {subset_samples} samples)...")
sfm_vals = calc_sfm(X_barr[:subset_samples])
meta_sub = df_meta.iloc[:subset_samples]

plt.figure(figsize=(10, 6))
plt.scatter(meta_sub['sinr_db'], sfm_vals, alpha=0.3, s=10, label="Samples", color='C0')
# Add a trend line (Moving Average or Lowess) to show the continuity
sns.regplot(x=meta_sub['sinr_db'], y=sfm_vals, 
            scatter=False, color='red', lowess=True, label="Trend (Lowess)")

plt.title("Barrage: Spectral Flatness vs Jamming Level")
plt.xlabel("SINR (dB)")
plt.ylabel("Spectral Flatness (0=Spiky, 1=Flat)")
plt.grid(True, alpha=0.3)
plt.legend()

save_plot("barrage_flatness_trend", machine_id="B",nb_id=NB_ID,fig_id="03")
plt.show()

# Fig 21.3 Feature Continuity: The "Inverted Bridge"

This plot confirms the **physical distinctness** of Barrage Jamming compared to the MIT Spot Jamming.

### The "White Noise" Trend 
* **Observation:** As we move Left (Lower SINR / More Noise), the dots move Up (Higher Spectral Flatness).
    * **High Noise (-15dB):** Flatness $\approx$ 1.0 (Maximum).
    * **Clean Signal (+20dB):** Flatness $\approx$ 0.1-0.3 (Low).
* **Physics Interpretation:**
    * **Barrage Jamming (Synthetic):** Fills the band with uniform energy. As it dominates the signal, the spectrum flattens.
    * **Contrast with Spot Jamming:** In the MIT data, high noise created a *spike* (Low Flatness).
* **Strategic Conclusion:** This proves our synthetic generator successfully creates "Broadband" interference, covering the "blind spot" that training solely on MIT data would have left.

In [ ]:
# Using a small offset (e.g., +2 or -2) ensures we find actual data 
# points near the edges without hitting the literal limit
target_min = MIT_SINR_MIN_DB + 2.0 
target_max = MIT_SINR_MAX_DB - 2.0

# Finds the index of the samples closest to our targets
idx_min = (df_meta['sinr_db'] - target_min).abs().idxmin()
idx_max = (df_meta['sinr_db'] - target_max).abs().idxmin()

val_min = df_meta.loc[idx_min, 'sinr_db']
val_max = df_meta.loc[idx_max, 'sinr_db']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# --- Row 1: Dataset Lower Bound (Noise Dominated) ---
ax_t_min, ax_f_min = axes[0, 0], axes[0, 1]

# Time Domain: Amplitude Analysis
ax_t_min.plot(X_barr[idx_min].real[:200], color='C0', label='Noisy Mixture')
ax_t_min.plot(Y_barr[idx_min].real[:200], color='C1', alpha=0.8, label='Clean Reference')
ax_t_min.set_title(f"Time Domain: Lower Bound ({val_min:.1f} dB SINR)")
ax_t_min.set_ylabel("Normalized Amplitude")
ax_t_min.legend(loc='upper right')

# Frequency Domain: PSD Analysis
ax_f_min.psd(Y_barr[idx_min], NFFT=1024, Fs=1.0, color='C1', label='Clean Signal')
ax_f_min.psd(X_barr[idx_min], NFFT=1024, Fs=1.0, color='C0', alpha=0.5, label='Noisy Mixture')
ax_f_min.set_title(f"Spectral Density: {val_min:.1f} dB (High Interference)")
ax_f_min.legend(loc='upper right')

# --- Row 2: Dataset Upper Bound (Signal Dominated) ---
ax_t_max, ax_f_max = axes[1, 0], axes[1, 1]

# Time Domain: Amplitude Analysis
ax_t_max.plot(X_barr[idx_max].real[:200], color='C0', label='Noisy Mixture')
ax_t_max.plot(Y_barr[idx_max].real[:200], color='C1', alpha=0.8, label='Clean Reference')
ax_t_max.set_title(f"Time Domain: Upper Bound ({val_max:.1f} dB SINR)")
ax_t_max.set_ylabel("Normalized Amplitude")
ax_t_max.set_xlabel("Sample Index (Time)")
ax_t_max.legend(loc='upper right')

# Frequency Domain: PSD Analysis
ax_f_max.psd(Y_barr[idx_max], NFFT=1024, Fs=1.0, color='C1', label='Clean Signal')
ax_f_max.psd(X_barr[idx_max], NFFT=1024, Fs=1.0, color='C0', alpha=0.5, label='Noisy Mixture')
ax_f_max.set_title(f"Spectral Density: {val_max:.1f} dB (Clear Signal)")
ax_f_max.set_xlabel("Normalized Frequency")
ax_f_max.legend(loc='upper right')

plt.tight_layout()

save_plot("barrage_jamming_examples", machine_id="B", nb_id=NB_ID, fig_id="04")
plt.show()

# Fig 21.4 Statistical Balance: Dataset Composition

The Statistical Balance analysis (Fig 21.4) performs the first critical sanity check: ensuring the dataset provides equal learning opportunities across all difficulty levels.

### The "Flat" Distribution
The histogram reveals a deliberate **Uniform Distribution** rather than a natural Gaussian curve.
* **Observation:** The bars maintain a roughly equal height across the entire SINR range.
* **Logic:** Unlike real-world data collection, where extreme events are rare, our synthetic generation creates an "Artificial Reality" where catastrophic jamming (-15 dB) is just as common as mild interference (+15 dB).

### Why "Natural" is Bad for Training
If we had relied on a "natural" distribution (a Bell Curve centered at 0 dB), the model would have failed:
* **The "Lazy Student" Problem:** The Autoencoder would optimize its weights primarily for the common cases (0 dB to +5 dB).
* **The Edge Case Failure:** It would treat Heavy Jamming (-15 dB) as a statistical outlier, failing to learn the aggressive denoising required to recover the signal.
* **Our Solution:** By forcing a flat distribution, we ensure the loss function penalizes errors in the "Hard" cases just as heavily as errors in the "Easy" cases.

### Operational Benefit
This uniformity is the backbone of the "Bilingual" strategy:
* **Machine B Reliability:** The Autoencoder will not "give up" when facing the harsh, wide-band noise of a Barrage attack.
* **Bias Prevention:** We explicitly prevent the model from biasing towards "clean" signals, ensuring it remains robust even when the jammer is significantly louder than the signal of interest.

In [ ]:
SAMPLE_LIMIT = 100_000 

# Calculate and Normalize Noise per-sample to avoid "Mixture of Gaussians" Kurtosis
noise_samples = []
# We take a subset of frames to keep it fast
num_frames = min(500, X_barr.shape[0]) 

for i in range(num_frames):
    # Extract noise for this specific frame
    frame_noise = X_barr[i] - Y_barr[i]
    
    # Normalize by the frame's own standard deviation
    # This aligns the 'width' of all Gaussians to 1.0
    std_dev = np.std(frame_noise)
    if std_dev > 0:
        normalized_frame = frame_noise / std_dev
        noise_samples.append(normalized_frame)

# Flatten and Sample
flat_noise = np.concatenate(noise_samples).ravel()
if len(flat_noise) > SAMPLE_LIMIT:
    indices = np.random.choice(len(flat_noise), size=SAMPLE_LIMIT, replace=False)
    sampled_noise = flat_noise[indices]
else:
    sampled_noise = flat_noise

# Stack Real/Imaginary components
data_to_plot = np.concatenate([np.real(sampled_noise), np.imag(sampled_noise)])

# Calculate Statistics (This should now be ~0.0)
k_val = stats.kurtosis(data_to_plot, fisher=True)

plt.figure(figsize=(8, 6))
sns.histplot(data_to_plot, bins=100, stat="density", color="C0", alpha=0.5, label="Normalized Noise", kde=True)

# Overlay Ideal Gaussian Curve (Standard Normal)
mu, std = stats.norm.fit(data_to_plot) 
xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = stats.norm.pdf(x, mu, std)
plt.plot(x, p, 'k', linewidth=2, label=f"Ideal Gaussian (μ={mu:.2f}, σ={std:.2f})")

plt.title(f"Normalized Noise Distribution (Excess Kurtosis = {k_val:.4f})")
plt.xlabel("Normalized Amplitude (Standard Deviations)")
plt.ylabel("Density")
plt.legend()
plt.grid(True, alpha=0.3)

save_plot("barrage_gaussianity_check_fixed", machine_id="B", nb_id=NB_ID, fig_id="05")
plt.show()

print(f"Kurtosis: {k_val:.4f} (Expected: ~0.0)")
if abs(k_val) > 0.5:
    print("WARNING: Noise distribution is deformed! Check normalization logic.")
else:
    print("SUCCESS: Noise is Gaussian.")

# Fig 21.5 Gaussianity & Noise Distribution Analysis

The Amplitude Distribution Analysis provides the statistical verification that the synthesized **Barrage Jamming** correctly simulates real-world Additive White Gaussian Noise (AWGN).

### 1. Statistical Verification of the Generator

Individual analysis of the synthetic noise slices confirms the mathematical integrity of the `generate_awgn` function.

* **Per-Sample Kurtosis:** Individual frames exhibit an average **Excess Kurtosis of ≈0.00**.
* **Implication:** The noise source is perfectly Gaussian. The absence of heavy tails or outliers in individual slices ensures the model learns to remove theoretically ideal interference.

### 2. Identifying the "Mixture of Gaussians" Effect

Initial aggregate analysis showed a Kurtosis of **3.11**. This is not a generator failure, but a characteristic of the dataset design.

* **Variance Mixing:** Because the dataset contains a wide range of SINR levels ( −12dB to 3dB ), the aggregate distribution is a **Mixture of Gaussians** with different variances.
* **The Result:** This naturally creates a leptokurtic (peaky) distribution. Normalizing the slices by their own standard deviation (as seen in the fixed plot) collapses this mixture back into a standard normal curve.

### 3. Outlier and Clipping Check

The maximum amplitude values observed (≈3.12σ) align perfectly with theoretical expectations for a sample size of $10^5$.

* **Verdict:** There is zero evidence of numerical clipping or saturation in the `mix_signal` logic. The full dynamic range of the interference is preserved.

### Final Verdict: Barrage Generation

The Barrage Jamming subset has successfully demonstrated:

1. **Mathematical Accuracy:** True AWGN generation verified by per-sample kurtosis.
2. **Dataset Diversity:** Validated variance mixture covering a  SINR range.
3. **Physical Integrity:** Clean signals (K≈−0.88) are successfully "masked" by the noise (K≈0.00) in the mixture, providing a robust training signal.

In [ ]:
print("--- DATA HYGIENE VERIFICATION ---")

# NaN/Inf Check (The "Training Killer")
has_nan = np.isnan(X_barr).any() or np.isnan(Y_barr).any()
has_inf = np.isinf(X_barr).any() or np.isinf(Y_barr).any()

print(f"Contains NaNs: {has_nan} (Must be False)")
print(f"Contains Infs: {has_inf} (Must be False)")

assert not has_nan, "CRITICAL: Dataset contains NaNs! Check generator division."
assert not has_inf, "CRITICAL: Dataset contains Infinities!"

# Dynamic Range Check (The "Gradient Exploder")
# We expect normalized data (approx 0.0 to 1.0 magnitude)
max_val_X = np.max(np.abs(X_barr))
max_val_Y = np.max(np.abs(Y_barr))

print(f"Max Input Magnitude:  {max_val_X:.4f} (Expected: ~1.0)")
print(f"Max Target Magnitude: {max_val_Y:.4f} (Expected: ~1.0)")

if max_val_X > 2.0:
    print("WARNING: Input data is not normalized! Gradients may explode.")
else:
    print("SUCCESS: Data range is safe for Neural Network.")

# Fig 21.6 Data Hygiene & Numerical Stability Report

The hygiene check ensures that the raw physics-based generation does not introduce numerical artifacts that could destabilize the training process.

### 1. Numerical Integrity (NaN/Inf)

The dataset was scanned for `NaN` (Not a Number) and `Inf` (Infinity) values.

* **Result:** No invalid values detected.
* **Verification:** This confirms the `mix_signal` logic is stable and there are no "Division by Zero" errors occurring during low-power noise scaling.

### 2. Dynamic Range Analysis

We measured the absolute magnitude of the complex I/Q samples.

* **Observed Max:** ~18.90.
* **Target Range:** < 1.0 (typical for Neural Network stability).
* **Findings:** The raw generation correctly preserves physical signal power, but the resulting magnitude is too high for direct ingestion into an Autoencoder architecture without causing gradient explosion.

### 3. Verdict: Two-Pass Scaling Strategy

To ensure the data remains calibrated across different interference types (Barrage vs. Spot), the following verdict was reached:

1. **Preserve Raw Physics:** The generation notebooks will save data in "Raw Volts" to maintain the integrity of the SNR/SINR calculations.
2. **Global Master Scaling:** After generating both Barrage and Spot datasets, a **Global Maximum** ($M$) will be calculated across the entire project.
3. **Unified Preprocessing:** All datasets will be scaled by this single value $M$ before training. This ensures that the neural network sees a consistent power relationship between different jamming categories.